In [ ]:
import duckdb
import pandas as pd
import os
import matplotlib.pyplot as plt

# Connect to our DuckDB database
db_path = os.path.abspath(os.path.join('..', 'bulgaria_energy.db'))
con = duckdb.connect(db_path)

# Set temp directory (Windows fix)
con.execute("SET temp_directory = 'C:/Users/stann/Projects/bulgaria-energy-platform/tmp'")

print("Connected to DuckDB successfully")

# Verify all our tables exist
tables = con.execute("SHOW TABLES").df()
print("\nAvailable tables:")
print(tables)


In [ ]:
# =============================================================================
# QUESTION 1: Monthly energy mix breakdown for 2024
# We want to see how each generation source changes month by month
# =============================================================================

monthly_mix = con.execute("""
    SELECT
        d.month,
        d.month_name,
        ROUND(AVG(g.nuclear_mw), 1)       AS nuclear_mw,
        ROUND(AVG(g.lignite_mw), 1)        AS lignite_mw,
        ROUND(AVG(g.solar_mw), 1)          AS solar_mw,
        ROUND(AVG(g.wind_onshore_mw), 1)   AS wind_mw,
        ROUND(AVG(g.hydro_reservoir_mw +
                  g.hydro_river_mw), 1)    AS hydro_mw,
        ROUND(AVG(g.gas_mw), 1)            AS gas_mw,
        ROUND(AVG(g.renewable_pct), 1)     AS renewable_pct,
        ROUND(AVG(g.total_generation_mw),1) AS total_mw
    FROM fct_generation_hourly g
    JOIN dim_date d ON g.date_key = d.date_key
    GROUP BY d.month, d.month_name
    ORDER BY d.month
""").df()

print(monthly_mix.to_string())

# Plot it
fig, ax = plt.subplots(figsize=(14, 6))

months = monthly_mix['month_name']
ax.stackplot(months,
    monthly_mix['nuclear_mw'],
    monthly_mix['lignite_mw'],
    monthly_mix['solar_mw'],
    monthly_mix['wind_mw'],
    monthly_mix['hydro_mw'],
    monthly_mix['gas_mw'],
    labels=['Nuclear', 'Lignite', 'Solar', 'Wind', 'Hydro', 'Gas'],
    colors=['#FF6B6B', '#4A4A4A', '#FFD93D', '#6BCB77', '#4D96FF', '#FF922B']
)

ax.set_title("Bulgaria Monthly Energy Mix 2024", fontsize=14)
ax.set_xlabel("Month")
ax.set_ylabel("Average Generation (MW)")
ax.legend(loc='upper left')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# QUESTION 2: How do electricity prices correlate with renewable share?
# This is the core question.
# =============================================================================

price_vs_renewable = con.execute("""
    WITH sofia_weather AS (
        SELECT date_key, shortwave_radiation_wm2
        FROM fct_weather_hourly
        WHERE city_key = 4
    )
    SELECT
        d.month,
        d.month_name,
        d.hour,
        g.renewable_pct,
        g.solar_mw,
        g.wind_onshore_mw,
        g.fossil_mw,
        g.total_generation_mw,
        p.price_eur_mwh,
        p.is_negative_price,
        w.shortwave_radiation_wm2
    FROM fct_generation_hourly g
    JOIN fct_prices_hourly p  ON g.date_key = p.date_key
    JOIN dim_date d           ON g.date_key = d.date_key
    JOIN sofia_weather w      ON g.date_key = w.date_key
""").df()

# --- Chart 1: Scatter plot — renewable % vs price ---
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left chart: scatter plot
axes[0].scatter(
    price_vs_renewable['renewable_pct'],
    price_vs_renewable['price_eur_mwh'],
    alpha=0.1,      # transparency so overlapping points are visible
    color='steelblue',
    s=5             # small dot size
)
axes[0].set_title("Renewable Share vs Electricity Price")
axes[0].set_xlabel("Renewable Generation (%)")
axes[0].set_ylabel("Price (€/MWh)")
axes[0].axhline(y=0, color='red', linestyle='--', linewidth=0.8)  # zero price line

# Right chart: average price by renewable % bucket
price_vs_renewable['renewable_bucket'] = (
    price_vs_renewable['renewable_pct'] // 10 * 10
).astype(int)

bucket_avg = price_vs_renewable.groupby('renewable_bucket')['price_eur_mwh'].mean().reset_index()

axes[1].bar(
    bucket_avg['renewable_bucket'],
    bucket_avg['price_eur_mwh'],
    width=8,
    color='steelblue',
    edgecolor='white'
)
axes[1].set_title("Average Price by Renewable Share Bucket")
axes[1].set_xlabel("Renewable Share (% bucket)")
axes[1].set_ylabel("Average Price (€/MWh)")

plt.tight_layout()
plt.show()

# Print correlation coefficient
correlation = price_vs_renewable['renewable_pct'].corr(
    price_vs_renewable['price_eur_mwh']
)
print(f"\nCorrelation between renewable share and price: {correlation:.3f}")
print("(Negative number = higher renewables → lower prices)")

# Count negative prices
neg_prices = price_vs_renewable['is_negative_price'].sum()
print(f"\nHours with negative prices in 2024: {neg_prices}")
print(f"That's {neg_prices/len(price_vs_renewable)*100:.1f}% of all hours")

In [ ]:
# =============================================================================
# QUESTION 3: Which hours of the day have highest/lowest prices?
# This reveals the daily demand cycle in Bulgaria's electricity market
# =============================================================================

hourly_prices = con.execute("""
    SELECT
        d.hour,
        ROUND(AVG(p.price_eur_mwh), 2)    AS avg_price,
        ROUND(MIN(p.price_eur_mwh), 2)     AS min_price,
        ROUND(MAX(p.price_eur_mwh), 2)     AS max_price,
        ROUND(STDDEV(p.price_eur_mwh), 2)  AS price_volatility
    FROM fct_prices_hourly p
    JOIN dim_date d ON p.date_key = d.date_key
    GROUP BY d.hour
    ORDER BY d.hour
""").df()

print(hourly_prices.to_string())

# --- Chart: Average price by hour with min/max range ---
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

hours = hourly_prices['hour']

# Left chart: average price by hour
axes[0].fill_between(hours,
    hourly_prices['min_price'].clip(lower=0),  # clip negatives to 0 for display
    hourly_prices['max_price'],
    alpha=0.15, color='steelblue', label='Min-Max range'
)
axes[0].plot(hours, hourly_prices['avg_price'],
    color='steelblue', linewidth=2.5, label='Average price')
axes[0].set_title("Average Electricity Price by Hour of Day")
axes[0].set_xlabel("Hour of Day")
axes[0].set_ylabel("Price (€/MWh)")
axes[0].set_xticks(range(0, 24))
axes[0].legend()
axes[0].axhline(y=hourly_prices['avg_price'].mean(),
    color='red', linestyle='--', linewidth=1, label='Overall average')

# Right chart: heatmap of price by hour and season
heatmap_data = con.execute("""
    SELECT
        d.season,
        d.hour,
        ROUND(AVG(p.price_eur_mwh), 2) AS avg_price
    FROM fct_prices_hourly p
    JOIN dim_date d ON p.date_key = d.date_key
    GROUP BY d.season, d.hour
    ORDER BY d.hour
""").df()

# Pivot for heatmap
heatmap_pivot = heatmap_data.pivot(
    index='season', columns='hour', values='avg_price'
)

# Reorder seasons logically
season_order = ['Winter', 'Spring', 'Summer', 'Autumn']
heatmap_pivot = heatmap_pivot.reindex(season_order)

im = axes[1].imshow(heatmap_pivot.values, aspect='auto', cmap='RdYlGn_r')
axes[1].set_title("Price Heatmap by Hour and Season")
axes[1].set_xlabel("Hour of Day")
axes[1].set_ylabel("Season")
axes[1].set_xticks(range(24))
axes[1].set_xticklabels(range(24))
axes[1].set_yticks(range(4))
axes[1].set_yticklabels(season_order)
plt.colorbar(im, ax=axes[1], label='€/MWh')

plt.tight_layout()
plt.show()

# Print key findings
print(f"\nCheapest hour on average: {hourly_prices.loc[hourly_prices['avg_price'].idxmin(), 'hour']}:00 "
      f"({hourly_prices['avg_price'].min():.2f} €/MWh)")
print(f"Most expensive hour on average: {hourly_prices.loc[hourly_prices['avg_price'].idxmax(), 'hour']}:00 "
      f"({hourly_prices['avg_price'].max():.2f} €/MWh)")
print(f"\nMost volatile hour (highest std dev): "
      f"{hourly_prices.loc[hourly_prices['price_volatility'].idxmax(), 'hour']}:00")

In [ ]:
# =============================================================================
# QUESTION 4: Solar radiation vs solar generation
# Does more sunshine actually produce more electricity?
# =============================================================================

solar_correlation = con.execute("""
    WITH sofia_weather AS (
        SELECT date_key, shortwave_radiation_wm2
        FROM fct_weather_hourly
        WHERE city_key = 4
    )
    SELECT
        w.shortwave_radiation_wm2,
        g.solar_mw,
        d.month,
        d.month_name
    FROM fct_generation_hourly g
    JOIN dim_date d       ON g.date_key = d.date_key
    JOIN sofia_weather w  ON g.date_key = w.date_key
    WHERE w.shortwave_radiation_wm2 > 0  -- daytime only
""").df()

fig, ax = plt.subplots(figsize=(10, 6))
scatter = ax.scatter(
    solar_correlation['shortwave_radiation_wm2'],
    solar_correlation['solar_mw'],
    c=solar_correlation['month'],  # color by month
    cmap='RdYlGn',
    alpha=0.3,
    s=5
)
plt.colorbar(scatter, ax=ax, label='Month')
ax.set_title("Solar Radiation vs Solar Generation — Sofia 2024")
ax.set_xlabel("Solar Radiation (W/m²)")
ax.set_ylabel("Solar Generation (MW)")
plt.tight_layout()
plt.show()

corr = solar_correlation['shortwave_radiation_wm2'].corr(solar_correlation['solar_mw'])
print(f"Correlation between solar radiation and solar generation: {corr:.3f}")

# =============================================================================
# QUESTION 5: Wind speed vs wind generation
# =============================================================================

wind_correlation = con.execute("""
    WITH sofia_weather AS (
        SELECT date_key, windspeed_10m_kmh
        FROM fct_weather_hourly
        WHERE city_key = 4
    )
    SELECT
        w.windspeed_10m_kmh,
        g.wind_onshore_mw,
        d.season
    FROM fct_generation_hourly g
    JOIN dim_date d       ON g.date_key = d.date_key
    JOIN sofia_weather w  ON g.date_key = w.date_key
""").df()

fig, ax = plt.subplots(figsize=(10, 6))
colors = {'Winter':'blue', 'Spring':'green', 'Summer':'orange', 'Autumn':'red'}
for season, group in wind_correlation.groupby('season'):
    ax.scatter(
        group['windspeed_10m_kmh'],
        group['wind_onshore_mw'],
        alpha=0.2, s=5,
        color=colors[season],
        label=season
    )
ax.set_title("Wind Speed vs Wind Generation 2024")
ax.set_xlabel("Wind Speed at 10m (km/h)")
ax.set_ylabel("Wind Generation (MW)")
ax.legend()
plt.tight_layout()
plt.show()

corr_wind = wind_correlation['windspeed_10m_kmh'].corr(wind_correlation['wind_onshore_mw'])
print(f"Correlation between wind speed and wind generation: {corr_wind:.3f}")

# =============================================================================
# QUESTION 6: Seasonal patterns in generation and prices
# =============================================================================

seasonal = con.execute("""
    SELECT
        d.season,
        ROUND(AVG(g.renewable_pct), 1)    AS avg_renewable_pct,
        ROUND(AVG(g.solar_mw), 1)         AS avg_solar_mw,
        ROUND(AVG(g.wind_onshore_mw), 1)  AS avg_wind_mw,
        ROUND(AVG(g.nuclear_mw), 1)       AS avg_nuclear_mw,
        ROUND(AVG(g.lignite_mw), 1)       AS avg_lignite_mw,
        ROUND(AVG(p.price_eur_mwh), 2)    AS avg_price,
        COUNT(CASE WHEN p.is_negative_price THEN 1 END) AS negative_price_hours
    FROM fct_generation_hourly g
    JOIN fct_prices_hourly p ON g.date_key = p.date_key
    JOIN dim_date d          ON g.date_key = d.date_key
    GROUP BY d.season
    ORDER BY avg_renewable_pct DESC
""").df()

print("\nSeasonal Summary:")
print(seasonal.to_string())

# =============================================================================
# QUESTION 7: Solar radiation comparison across cities
# =============================================================================

city_solar = con.execute("""
    SELECT
        c.city,
        c.region,
        d.month_name,
        d.month,
        ROUND(AVG(w.shortwave_radiation_wm2), 1) AS avg_radiation
    FROM fct_weather_hourly w
    JOIN dim_city c ON w.city_key = c.city_key
    JOIN dim_date d ON w.date_key = d.date_key
    GROUP BY c.city, c.region, d.month_name, d.month
    ORDER BY c.city, d.month
""").df()

fig, ax = plt.subplots(figsize=(14, 6))
colors_city = {
    'Sofia':'blue', 'Plovdiv':'green',
    'Varna':'red', 'Burgas':'orange', 'Pleven':'purple'
}
for city, group in city_solar.groupby('city'):
    ax.plot(
        group['month_name'],
        group['avg_radiation'],
        marker='o', linewidth=2,
        label=city, color=colors_city[city]
    )
ax.set_title("Monthly Solar Radiation by City 2024")
ax.set_xlabel("Month")
ax.set_ylabel("Average Solar Radiation (W/m²)")
ax.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
con.close()